In [1]:
import pandas as pd 

df = pd.read_csv("/home/jovyan/Exploring-LLM-Based-Feature-Extraction-for-Depression-Assessment-from-Therapy-Transcripts/OUTPUT/10_synthetic_transcripts_length_controlled_merged_turns.csv")
df.head(1)

,synthetic_id,PHQ8_Concentrating,PHQ8_Appetite,PHQ8_Depressed,PHQ8_Tired,PHQ8_NoInterest,PHQ8_Failure,PHQ8_Moving,PHQ8_Sleep,target_PHQ8_Score,...,target_participant_turns,target_participant_words,target_total_turns,target_mean_words_per_participant_turn,target_median_words_per_participant_turn,length_sampling_mode,persona,few_shot_participants,synthetic_transcript,messages
0,syn_0000,3,1,3,2,2,3,1,2,17,...,52,900,104,46.826923,19.0,neighbor_sample_clipped,"{""gender"": ""non-binary"", ""first_name"": ""Wyatt""...","[405, 367, 384]","Therapist | Good afternoon, Wyatt. How are you...","[{""role"": ""system"", ""content"": ""You generate s..."


In [2]:
from __future__ import annotations
from typing import List, Dict, Any, Tuple
import pandas as pd

FeatureNames = [
    "target_PHQ8_Concentrating",
    "target_PHQ8_Appetite",
    "target_PHQ8_Depressed",
    "target_PHQ8_Tired",
    "target_PHQ8_NoInterest",
    "target_PHQ8_Failure",
    "target_PHQ8_Moving",
    "target_PHQ8_Sleep",
]

def _parse_line(line: str, line_no: int) -> Tuple[str, str, List[int]]:
    parts = [p.strip() for p in line.split("|")]

    if len(parts) != 10:
        raise ValueError(
            f"Zeile {line_no}: Erwartete 10 Spalten (Speaker, Utterance + 8 Werte), "
            f"erhielt aber {len(parts)} → {parts!r}"
        )

    speaker, utterance = parts[0], parts[1]

    try:
        values = [int(v) for v in parts[2:]]
    except ValueError as exc:
        raise ValueError(
            f"Zeile {line_no}: Mindestens ein Feature‑Wert ist keine ganze Zahl – {exc}"
        ) from exc

    return speaker, utterance, values

def parse_transcript(
    raw_text: str,
    as_dataframe: bool = True,
) -> List[Dict[str, Any]] | pd.DataFrame:

    rows: List[Dict[str, Any]] = []

    for i, line in enumerate(raw_text.splitlines(), start=1):
        line = line.strip()
        if not line:                # leere Zeile → überspringen
            continue

        speaker, utterance, values = _parse_line(line, i)

        row = {
            "speaker":    speaker,
            "utterance":  utterance,
        }
        row.update(dict(zip(FeatureNames, values)))
        rows.append(row)

    if as_dataframe:
        return pd.DataFrame(rows)
    return rows

In [3]:
parsed_chunks = []

for idx, row in df.iterrows():
    try:
        df_parsed = parse_transcript(row["synthetic_transcript"])
    except Exception as e:
        print(f"Fehler beim Parsen von synthetic_id={row['synthetic_id']}: {e}")
        continue

    meta_cols = [c for c in df.columns if c != "synthetic_transcript"]
    for col in meta_cols:
        df_parsed[col] = row[col]

    parsed_chunks.append(df_parsed)

if parsed_chunks:
    df_final = pd.concat(parsed_chunks, ignore_index=True)
else:
    raise RuntimeError("Keine Transkripte konnten erfolgreich geparst werden.")

desired_order = (
    ["synthetic_id", "persona"]
    + ["speaker", "utterance"]
    + FeatureNames
)
df_final = df_final[[c for c in desired_order if c in df_final.columns]]

print("\n=== Vorschau des geparsten DataFrames ===")
display(df_final.head(1))

print("\nGesamtzahl Zeilen (einzelne Turns):", len(df_final))

Fehler beim Parsen von synthetic_id=syn_0000: Zeile 1: Erwartete 10 Spalten (Speaker, Utterance + 8 Werte), erhielt aber 2 → ['Therapist', 'Good afternoon, Wyatt. How are you feeling today?']
Fehler beim Parsen von synthetic_id=syn_0003: Zeile 1: Erwartete 10 Spalten (Speaker, Utterance + 8 Werte), erhielt aber 2 → ['Therapist', 'How are you feeling today, Benjamin?']
Fehler beim Parsen von synthetic_id=syn_0004: Zeile 1: Erwartete 10 Spalten (Speaker, Utterance + 8 Werte), erhielt aber 2 → ['Therapist', 'good morning nora how are you feeling today']
Fehler beim Parsen von synthetic_id=syn_0005: Zeile 1: Erwartete 10 Spalten (Speaker, Utterance + 8 Werte), erhielt aber 2 → ['Therapist', 'Good morning, Audrey. How are you feeling today?']


RuntimeError: Keine Transkripte konnten erfolgreich geparst werden.

In [4]:
df_final.to_csv("/home/jovyan/finetuning-qwen-datavol-1/synthetic_data_generation/output/parsed_9_synthetic_transcripts.csv")